# Prepare Data

One-shot data preparation notebook. Runs phases 1–3 of the pipeline:

1. **Symbol screening** — filter ru1000 to symbols with sufficient news coverage
2. **Price data** — fetch 2018-2020 daily OHLCV bars from Alpaca for all screened symbols
3. **Fundamentals** — fetch yfinance snapshots for all symbols

Each phase is resumable: already-cached symbols are skipped.
Final universe is saved to `data/symbols.yml`.

In [1]:
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd
import yaml

import sentiment  # triggers load_dotenv()
from sentiment.log import setup_logging
from sentiment.features.screening import screen_by_coverage
from sentiment.sources.alpaca import AlpacaSource
from sentiment.sources.cache import MarketDataCache
from sentiment.sources.fundamental import FundamentalCache, FundamentalSource
from sentiment.sources.news.repository import ArticleRepository

setup_logging()
logger = logging.getLogger("prepare_data")

In [2]:
# --- Config ---

NEWS_START = (2018, 1)
NEWS_END   = (2020, 6)
PRICE_YEARS = [2018, 2019, 2020]
MIN_AVG_ARTICLES = 5.0   # minimum average articles/month for a symbol to pass screening
ALPACA_BATCH_SIZE = 50   # symbols per Alpaca API call

ROOT        = Path("..").resolve()
DATA_DIR    = ROOT / "data"
RU1000_PATH = DATA_DIR / "ru1000.yml"
SYMBOLS_PATH = DATA_DIR / "symbols.yml"

## Phase 1 — Symbol Screening

Filter the 992 ru1000 symbols down to those with sufficient news coverage
in the 2018-01 → 2020-06 corpus.

In [3]:
with open(RU1000_PATH) as f:
    ru1000: dict[str, str] = yaml.safe_load(f)["companies"]

print(f"ru1000 universe: {len(ru1000)} symbols")

ru1000 universe: 992 symbols


In [4]:
repo = ArticleRepository()

coverage_df = screen_by_coverage(
    repo,
    list(ru1000.keys()),
    start=NEWS_START,
    end=NEWS_END,
    min_avg_articles=MIN_AVG_ARTICLES,
)

candidates: list[str] = coverage_df[coverage_df["passes"]]["ticker"].tolist()

print(f"\nSymbols passing news coverage screen: {len(candidates)} / {len(ru1000)}")
print(f"(min_avg_articles={MIN_AVG_ARTICLES}, period {NEWS_START} → {NEWS_END})")
print()
print("Top 30 by average articles/month:")
print(
    coverage_df[coverage_df["passes"]]
    .sort_values("avg_articles_per_month", ascending=False)
    .head(30)
    .to_string(index=False)
)

13:37:24 INFO     sentiment.features.screening  Coverage screen: 614/992 tickers pass (min_avg=5.0 over 30 months)



Symbols passing news coverage screen: 614 / 992
(min_avg_articles=5.0, period (2018, 1) → (2020, 6))

Top 30 by average articles/month:
ticker  months_tracked  avg_articles_per_month  passes
  GILD              30                   98.50    True
   JPM              30                   95.03    True
  AVGO              30                   82.93    True
  INTC              30                   81.93    True
    HD              30                   80.67    True
   FDX              30                   76.03    True
   PCG              30                   74.67    True
  NFLX              30                   65.47    True
   OXY              30                   64.03    True
  TSLA              30                   62.17    True
   LLY              30                   62.10    True
   LUV              30                   60.77    True
   MDT              30                   58.80    True
   LOW              30                   58.77    True
 GOOGL              30                

## Phase 2 — Fetch Price Data

Fetch daily OHLCV bars from Alpaca for 2018, 2019, 2020.
Symbols already cached for a given year are skipped.
Symbols that return no bars across all three years are dropped from the universe.

In [5]:
alpaca = AlpacaSource()
price_cache = MarketDataCache()

# Track number of years with successful data per symbol
symbol_success: dict[str, int] = {s: 0 for s in candidates}

for year in PRICE_YEARS:
    start_dt = datetime(year, 1, 1)
    end_dt   = datetime(year, 12, 31)

    # Separate already-cached symbols from those that still need fetching
    to_fetch: list[str] = []
    for symbol in candidates:
        try:
            price_cache.load(symbol, year)
            symbol_success[symbol] += 1  # cached == previously successful fetch
        except FileNotFoundError:
            to_fetch.append(symbol)

    n_cached = len(candidates) - len(to_fetch)
    if not to_fetch:
        logger.info("Year %d: all %d symbols already cached — skipping", year, n_cached)
        continue

    logger.info(
        "Year %d: fetching %d symbols (%d already cached)",
        year, len(to_fetch), n_cached,
    )

    for i in range(0, len(to_fetch), ALPACA_BATCH_SIZE):
        batch = to_fetch[i : i + ALPACA_BATCH_SIZE]
        batch_label = f"{batch[0]}…{batch[-1]}" if len(batch) > 1 else batch[0]
        logger.info("  Year %d batch %d/%d: %s", year, i // ALPACA_BATCH_SIZE + 1,
                    -(-len(to_fetch) // ALPACA_BATCH_SIZE), batch_label)

        try:
            df = alpaca.fetch_bars(batch, "1Day", start_dt, end_dt)
        except RuntimeError as exc:
            logger.error("Alpaca fetch failed for batch [%s]: %s", batch_label, exc)
            continue

        for symbol in batch:
            sym_df = (
                df[df["symbol"] == symbol].drop(columns="symbol")
                if not df.empty
                else pd.DataFrame()
            )
            if sym_df.empty:
                logger.warning("No bars returned for %s in %d", symbol, year)
            else:
                price_cache.store(symbol, year, sym_df)
                symbol_success[symbol] += 1
                logger.debug("Stored %d bars for %s %d", len(sym_df), symbol, year)

print("\nPrice data fetch complete.")

13:37:25 INFO     prepare_data  Year 2018: fetching 1 symbols (613 already cached)
13:37:25 INFO     prepare_data    Year 2018 batch 1/1: CCC
13:37:25 WARNING  prepare_data  No bars returned for CCC in 2018
13:37:26 INFO     prepare_data  Year 2019: fetching 1 symbols (613 already cached)
13:37:26 INFO     prepare_data    Year 2019 batch 1/1: CCC
13:37:26 WARNING  prepare_data  No bars returned for CCC in 2019
13:37:26 INFO     prepare_data  Year 2020: all 614 symbols already cached — skipping



Price data fetch complete.


In [6]:
# Build final universe: keep symbols with price data for at least 1 year
universe: dict[str, str] = {
    t: ru1000[t]
    for t in candidates
    if symbol_success[t] > 0
}

n_dropped = len(candidates) - len(universe)
print(f"Final universe: {len(universe)} symbols")
print(f"Dropped (no Alpaca data): {n_dropped}")

if n_dropped > 0:
    dropped = [t for t in candidates if symbol_success[t] == 0]
    print(f"  {dropped}")

# Year coverage breakdown
coverage_counts = {1: 0, 2: 0, 3: 0}
for t in universe:
    coverage_counts[symbol_success[t]] = coverage_counts.get(symbol_success[t], 0) + 1
print("\nYears of price data per symbol:")
for n_years, count in sorted(coverage_counts.items()):
    print(f"  {n_years} year(s): {count} symbols")

# Save to symbols.yml
with open(SYMBOLS_PATH, "w") as f:
    yaml.dump({"companies": universe}, f, default_flow_style=False, allow_unicode=True)
print(f"\nSaved universe to {SYMBOLS_PATH}")

Final universe: 614 symbols
Dropped (no Alpaca data): 0

Years of price data per symbol:
  1 year(s): 1 symbols
  2 year(s): 0 symbols
  3 year(s): 613 symbols

Saved universe to /Users/plouc314/Documents/finance/quant-sentiment-score/data/symbols.yml


## Phase 3 — Fetch Fundamentals

Fetch a yfinance snapshot for each symbol in the universe.
Symbols already present in `data/fundamentals/fundamentals.csv` are skipped.

In [7]:
fund_source = FundamentalSource(request_delay=1.5)
fund_cache  = FundamentalCache()

# Determine which symbols still need fetching
already_cached: set[str] = set(fund_cache.load_all().index.tolist())
to_fetch_fund = [s for s in universe if s not in already_cached]

print(f"Symbols already cached: {len(already_cached)}")
print(f"Symbols to fetch:       {len(to_fetch_fund)}")

if to_fetch_fund:
    logger.info("Fetching fundamentals for %d symbols", len(to_fetch_fund))
    results = fund_source.fetch_many(to_fetch_fund)
    for symbol, data in results.items():
        fund_cache.store(symbol, data)
    print(f"\nFetched and stored fundamentals for {len(results)} symbols.")
else:
    print("\nAll fundamentals already cached — nothing to fetch.")

Symbols already cached: 617
Symbols to fetch:       0

All fundamentals already cached — nothing to fetch.


## Summary

In [8]:
fund_all = fund_cache.load_all()
fund_covered = set(fund_all.index) & set(universe)
n_missing_fund = len(universe) - len(fund_covered)

print("=" * 50)
print("DATA PREPARATION SUMMARY")
print("=" * 50)
print(f"Universe:                   {len(universe)} symbols")
print(f"Symbols with price data:    {len(universe)} (all)")
print(f"Symbols with fundamentals:  {len(fund_covered)} ({n_missing_fund} missing)")
print()
print("Price data coverage by year:")
for year in PRICE_YEARS:
    n = sum(
        1 for s in universe
        if (DATA_DIR / "historical-prices" / s / f"{year}.csv").exists()
    )
    print(f"  {year}: {n} / {len(universe)} symbols")
print()
print(f"Universe saved to: {SYMBOLS_PATH}")
print(f"Next step: run Phase 4 (sentiment encoding) in prepare_data_sentiment.ipynb")

DATA PREPARATION SUMMARY
Universe:                   614 symbols
Symbols with price data:    614 (all)
Symbols with fundamentals:  614 (0 missing)

Price data coverage by year:
  2018: 613 / 614 symbols
  2019: 613 / 614 symbols
  2020: 614 / 614 symbols

Universe saved to: /Users/plouc314/Documents/finance/quant-sentiment-score/data/symbols.yml
Next step: run Phase 4 (sentiment encoding) in prepare_data_sentiment.ipynb


## Phase 4 — Sentiment Encoding

Run BART summarisation + FinBERT encoding for every symbol in the universe.
Results are cached per-symbol to `data/sentiment/<SYMBOL>.parquet` — already-encoded
symbols are skipped so the loop is fully resumable.

In [12]:
from sentiment.embeddings import SentimentPipeline, SentimentCache

DEVICE = "cpu"

sentiment_cache = SentimentCache()
pipeline = SentimentPipeline(device=DEVICE)

n_done = sum(1 for s in universe if sentiment_cache.exists(s))
n_total = len(universe)
print(f"Already encoded: {n_done} / {n_total} symbols")

/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Already encoded: 44 / 614 symbols


In [14]:
for i, symbol in enumerate(universe, 1):
    if sentiment_cache.exists(symbol):
        continue
    articles = repo.read_symbol(symbol, start=NEWS_START, end=NEWS_END)
    if not articles:
        logger.warning("[%d/%d] %s: no articles — skipping", i, n_total, symbol)
        continue
    df = pipeline.process_ticker_articles({symbol: articles})
    sentiment_cache.store(symbol, df)
    logger.info("[%d/%d] %s: encoded %d days", i, n_total, symbol, len(df))

n_encoded = sum(1 for s in universe if sentiment_cache.exists(s))
print(f"\nSentiment encoding complete: {n_encoded} / {n_total} symbols cached")

15:14:18 INFO     prepare_data  [512/614] UGI: encoded 219 days
15:14:24 INFO     prepare_data  [513/614] ARMK: encoded 177 days
15:14:27 INFO     prepare_data  [514/614] VMI: encoded 129 days
15:14:42 INFO     prepare_data  [515/614] BURL: encoded 360 days
15:14:50 INFO     prepare_data  [516/614] TRGP: encoded 319 days
15:14:55 INFO     prepare_data  [517/614] FLEX: encoded 176 days
15:15:11 INFO     prepare_data  [518/614] LEN: encoded 458 days
15:15:17 INFO     prepare_data  [519/614] POST: encoded 194 days
15:15:25 INFO     prepare_data  [520/614] RF: encoded 315 days
15:15:36 INFO     prepare_data  [521/614] CIEN: encoded 312 days
15:15:55 INFO     prepare_data  [522/614] CNC: encoded 490 days
15:16:00 INFO     prepare_data  [523/614] AIT: encoded 182 days
15:16:10 INFO     prepare_data  [524/614] ROP: encoded 358 days
15:16:46 INFO     prepare_data  [525/614] PCG: encoded 532 days
15:16:52 INFO     prepare_data  [526/614] AXTA: encoded 220 days
15:16:57 INFO     prepare_data  [5


Sentiment encoding complete: 614 / 614 symbols cached


## Phase 5 — Train / Test Split

Assign each symbol a deterministic per-symbol train/test cutoff date, staggered
around a nominal date to avoid concentrating a single market regime in the test
set. Additionally hold out 20 % of symbols entirely (stratified by news coverage
tier) to measure cross-symbol generalisation separately.

In [9]:
import hashlib
import random

NOMINAL_CUTOFF = pd.Timestamp("2019-10-01")
STAGGER_DAYS   = 45   # each symbol's cutoff = NOMINAL_CUTOFF ± U[-45, 45] days
HELD_OUT_FRAC  = 0.20 # fraction of symbols held out of training entirely
VAL_MONTHS     = 2    # months immediately before cutoff reserved for validation
SPLIT_SEED     = 42

SPLITS_PATH = DATA_DIR / "splits.yml"

In [10]:
# --- Stratified held-out split ---
#
# Divide symbols into three news-coverage tiers so that both the training
# pool and the held-out pool have a similar coverage distribution.

def _coverage_tier(avg: float) -> str:
    if avg >= 30:
        return "high"
    elif avg >= 10:
        return "medium"
    return "low"


# Filter coverage_df to the final universe only
universe_set = set(universe)
cov = (
    coverage_df[coverage_df["ticker"].isin(universe_set)]
    .set_index("ticker")["avg_articles_per_month"]
)

tiers: dict[str, list[str]] = {"high": [], "medium": [], "low": []}
for symbol in universe:
    tiers[_coverage_tier(cov[symbol])].append(symbol)

print("Coverage tier sizes:")
for t, syms in tiers.items():
    print(f"  {t:6s}: {len(syms)} symbols")

# Within each tier take a deterministic 20 % as held-out
rng = random.Random(SPLIT_SEED)
train_symbols: list[str] = []
held_out_symbols: list[str] = []

for tier_syms in tiers.values():
    ordered = sorted(tier_syms)  # stable sort before shuffle
    rng.shuffle(ordered)
    n_held = max(1, round(len(ordered) * HELD_OUT_FRAC))
    held_out_symbols.extend(ordered[:n_held])
    train_symbols.extend(ordered[n_held:])

print(f"\nTrain symbols:     {len(train_symbols)}")
print(f"Held-out symbols:  {len(held_out_symbols)}")

Coverage tier sizes:
  high  : 88 symbols
  medium: 399 symbols
  low   : 127 symbols

Train symbols:     491
Held-out symbols:  123


In [11]:
# --- Per-symbol cutoff dates ---
#
# Each symbol gets a deterministic offset derived from its ticker string so
# that re-running the cell always produces the same splits.yml.

def _symbol_cutoff(symbol: str) -> pd.Timestamp:
    h = int(hashlib.md5(symbol.encode()).hexdigest(), 16)
    offset = (h % (2 * STAGGER_DAYS + 1)) - STAGGER_DAYS
    return NOMINAL_CUTOFF + pd.Timedelta(days=int(offset))


cutoffs = {s: _symbol_cutoff(s).strftime("%Y-%m-%d") for s in universe}

# Sanity-check: show the distribution of cutoff dates
cutoff_ts = pd.Series(pd.to_datetime(list(cutoffs.values())))
print("Cutoff date distribution:")
print(f"  min : {cutoff_ts.min().date()}")
print(f"  max : {cutoff_ts.max().date()}")
print(f"  mean: {cutoff_ts.mean().date()}")

# --- Save splits.yml ---
splits = {
    "config": {
        "nominal_cutoff": NOMINAL_CUTOFF.strftime("%Y-%m-%d"),
        "stagger_days":   STAGGER_DAYS,
        "val_months":     VAL_MONTHS,
        "held_out_frac":  HELD_OUT_FRAC,
        "seed":           SPLIT_SEED,
    },
    "train_symbols":    sorted(train_symbols),
    "held_out_symbols": sorted(held_out_symbols),
    "cutoffs":          cutoffs,
}

with open(SPLITS_PATH, "w") as f:
    yaml.dump(splits, f, default_flow_style=False, allow_unicode=True, sort_keys=True)

print(f"\nSaved splits to {SPLITS_PATH}")

Cutoff date distribution:
  min : 2019-08-17
  max : 2019-11-15
  mean: 2019-09-30

Saved splits to /Users/plouc314/Documents/finance/quant-sentiment-score/data/splits.yml
